# Mistral & Mixtral Setup - Secure Colab Guide

**Run Mistral AI models with 100% privacy**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Models Available

- **Mistral 7B** - 7B parameters, excellent performance
- **Mistral 7B Instruct** - Fine-tuned for chat
- **Mixtral 8x7B** - Mixture of Experts, sparse architecture
- **Mixtral 8x22B** - Larger MoE variant

---


In [ ]:
# Configuration
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # @param ["mistralai/Mistral-7B-v0.3", "mistralai/Mistral-7B-Instruct-v0.2", "mistralai/Mixtral-8x7B-v0.1", "mistralai/Mixtral-8x7B-Instruct-v0.1"]

USE_QUANTIZATION = True  # @param {type:"boolean"}
QUANT_BITS = 4  # @param [4, 8]

print(f"\ud83d\udc49 Model: {MODEL_ID}")
print(f"\ud83d\udc49 Quantization: {QUANT_BITS}-bit" if USE_QUANTIZATION else "\ud83d\udc49 Quantization: None")

In [ ]:
# Security environment
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/.hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!mkdir -p /content/.hf_cache

# Install dependencies
!pip install -q transformers accelerate bitsandbytes hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("\u2705 Environment configured!")

In [ ]:
import torch

# Check GPU
if torch.cuda.is_available():
    print(f"\u2705 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Memory optimization
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
else:
    print("\u274c No GPU - CPU mode")

In [ ]:
# Load Mistral
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"\nLoading {MODEL_ID}...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
print("\u2705 Tokenizer loaded")

# Model with quantization
if USE_QUANTIZATION and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=QUANT_BITS == 4,
        load_in_8bit=QUANT_BITS == 8,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" if torch.cuda.is_available() else "cpu"
    )

model.eval()
print("\u2705\u2705 Model loaded!\u2705\u2705")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Chat template for Mistral
def chat(prompt, max_tokens=256):
    """Generate response with Mistral."""
    messages = [{"role": "user", "content": prompt}]
    
    # Apply chat template
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

print("\u2705 Chat function ready!")

# Test
print("\nTest:")
result = chat("What is machine learning?", max_tokens=100)
print(f"\ud83d\udcac {result}")

In [ ]:
# Memory cleanup
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\u2705 Memory cleared!")

---

**Models**: [Mistral on HuggingFace](https://huggingface.co/mistralai)

**Star**: [GitHub](https://github.com/unn-Known1/huggingface-colab-secure-setup)